# Laboratori 08: Probabiliteti dhe përhapja e pacaktueshmërisë

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si kalojmë nga një realizim i rastit te një përfundim sasior me gabim të raportuar?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- llogarisë pritjen dhe variancën e ndryshoreve të thjeshta;
- verifikojë ligjin e numrave të mëdhenj dhe teoremën qendrore kufitare;
- përhapë pacaktueshmërinë me linearizim dhe Monte Carlo;
- nxjerrë kompleksitetin $\Theta(N)$ të vlerësuesve;
- përdorë algoritmin e Welford-it për variancë të qëndrueshme;
- dallojë devijimin standard nga gabimi standard.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Probabiliteti nuk përdoret këtu si sinonim i mungesës së njohurive. Ai është një model sasior për rezultate të shumëfishta ose për pacaktueshmëri të përcaktuar. Duhet deklaruar qartë se cila madhësi është e rastit, cila shpërndarje supozohet dhe nëse mostrat janë të pavarura.

Një realizim nuk është një shpërndarje dhe një histogram nuk është provë e vetme. Përfundimet ndërtohen nga pritja, varianca, kovarianca, gabimi standard dhe kontrollet e përsëritjes. Në çdo qelizë pyetni nëse po përshkruani luhatjen e individëve apo pacaktueshmërinë e një vlerësuesi.


## 1. Nga rezultatet e mundshme te madhësitë përmbledhëse

Për një ndryshore diskrete $X$ me vlera $x_i$ dhe probabilitete $p_i$,

$$\mathbb E[X]=\sum_i x_ip_i,$$

$$\operatorname{Var}(X)=\mathbb E[(X-\mu)^2]=\mathbb E[X^2]-\mu^2.$$

Devijimi standard $\sigma$ përshkruan shpërndarjen e vlerave individuale. Gabimi standard i mesatares,

$$\operatorname{SE}(\bar X)=\frac{\sigma}{\sqrt N},$$

përshkruan luhatjen e mesatares së $N$ mostrave të pavarura. Këto dy madhësi nuk janë të këmbyeshme.


## 2. Shembull me dorë: ndryshorja Bernoulli

Le të jetë $X=1$ me probabilitet $p$ dhe $X=0$ me probabilitet $1-p$. Atëherë

$$\mathbb E[X]=p,$$

$$\mathbb E[X^2]=p,$$

$$\operatorname{Var}(X)=p-p^2=p(1-p).$$

Për shumën $S_N=X_1+\cdots+X_N$ të provave të pavarura,

$$\mathbb E[S_N]=Np,\qquad \operatorname{Var}(S_N)=Np(1-p).$$

Për mesataren $\bar X=S_N/N$, varianca është $p(1-p)/N$. Këtu del drejtpërdrejt ligji $N^{-1/2}$.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
p=0.3; N=20
x=(rng.random(N)<p).astype(int)
print('mostra:',x)
print('mesatarja=',x.mean(),'pritja=',p)
print('varianca e mostrës=',x.var(ddof=1),'teorike=',p*(1-p))


## 3. Ligji i numrave të mëdhenj

Për mostra të pavarura me pritje të fundme, mesatarja kumulative afrohet te $\mu$ kur $N$ rritet. Kjo nuk do të thotë se afrohet në mënyrë monotone. Trajektorja luhatet, por amplituda tipike e luhatjeve ulet si $N^{-1/2}$.


In [ ]:
N=20_000
x=(rng.random(N)<p).astype(float)
n=np.arange(1,N+1)
mes_kum=np.cumsum(x)/n
se=np.sqrt(p*(1-p)/n)

fig,ax=plt.subplots(figsize=(7,4))
ax.semilogx(n,mes_kum,label='mesatarja kumulative')
ax.axhline(p,color='k',ls='--',label='p')
ax.fill_between(n,p-2*se,p+2*se,alpha=.2,label=r'$p\pm2SE$')
ax.set(xlabel='N',ylabel='mesatarja'); ax.grid(alpha=.3); ax.legend(); plt.show()


## 4. Teorema qendrore kufitare

Për $N$ mjaft të madh,

$$Z=\frac{\bar X-\mu}{\sigma/\sqrt N}$$

afrohet me shpërndarjen normale standarde, nën kushte të përgjithshme. Teorema flet për shpërndarjen e **mesatareve të përsëritura**, jo domosdoshmërisht për shpërndarjen e vetë $X$.


In [ ]:
M=10_000; N=40
mesatare=np.mean(rng.exponential(scale=2.0,size=(M,N)),axis=1)
z=(mesatare-2.0)/(2.0/np.sqrt(N))
print('mesatarja e z=',z.mean(),'devijimi=',z.std(ddof=1))
plt.hist(z,bins=50,density=True,alpha=.7,label='mesataret e standardizuara')
xx=np.linspace(-4,4,400); plt.plot(xx,np.exp(-xx**2/2)/np.sqrt(2*np.pi),'k',label='N(0,1)')
plt.grid(alpha=.3); plt.legend(); plt.show()


## 5. Përhapja lineare e pacaktueshmërisë

Për $Y=f(X_1,\ldots,X_d)$, linearizimi rreth mesatareve jep

$$\operatorname{Var}(Y)\approx\nabla f^T\Sigma\nabla f,$$

ku $\Sigma$ është matrica e kovariancës. Për ndryshore të pavarura,

$$\sigma_Y^2\approx\sum_i\left(\frac{\partial f}{\partial x_i}\right)^2\sigma_i^2.$$

Shembull: $R=V/I$. Me $V=10.0\pm0.2$ V dhe $I=2.0\pm0.05$ A,

$$\sigma_R^2\approx\left(\frac{1}{I}\sigma_V\right)^2+left(\frac{V}{I^2}\sigma_I\right)^2.$$


In [ ]:
V,sV=10.0,0.2; I,sI=2.0,0.05
R=V/I
sR=np.sqrt((sV/I)**2+(V*sI/I**2)**2)
print(f'R = {R:.4f} ± {sR:.4f} ohm')

M=300_000
Vs=rng.normal(V,sV,M); Is=rng.normal(I,sI,M); Rs=Vs/Is
print('Monte Carlo:',Rs.mean(),Rs.std(ddof=1))


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 6.1 Kompleksiteti i mesatares dhe Monte Carlo-s

Për $N$ vlera, shuma kërkon $N-1$ mbledhje dhe një pjesëtim:

$$T_{\bar X}(N)=aN+b\in\Theta(N).$$

Nëse ruhen të gjitha mostrat, kujtesa është $\Theta(N)$. Algoritmet rrjedhëse për mesataren dhe variancën përdorin $\Theta(1)$ kujtesë. Për $d$ madhësi dhe $N$ realizime, një model me kosto $C(d)$ për realizim kërkon $\Theta(NC(d))$.

Gjysmëzimi i gabimit standard kërkon katërfishim të $N$ sepse $SE\propto N^{-1/2}$.


In [ ]:
Ns=np.array([10_000,20_000,40_000,80_000,160_000,320_000]); ts=[]
for N in Ns:
    x=rng.normal(size=N)
    t0=perf_counter(); x.mean(); ts.append(perf_counter()-t0)
print('pjerrësia e matur=',np.polyfit(np.log(Ns),np.log(ts),1)[0])


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 7.1 Pse formula $E[X^2]-E[X]^2$ mund të dështojë

Kur vlerat janë të mëdha dhe varianca e vogël, dy termat $E[X^2]$ dhe $E[X]^2$ janë pothuajse të barabartë. Zbritja e tyre humbet shifra. Algoritmi i Welford-it përditëson mesataren dhe shumën e devijimeve pa këtë anulim:

$$\delta=x_n-\mu_{n-1},\qquad \mu_n=\mu_{n-1}+\frac{\delta}{n},$$

$$M_{2,n}=M_{2,n-1}+\delta(x_n-\mu_n).$$

Në fund $s^2=M_2/(N-1)$.


In [ ]:
def welford(v):
    n=0; mean=0.0; M2=0.0
    for x in v:
        n+=1; delta=x-mean; mean+=delta/n; M2+=delta*(x-mean)
    return mean,M2/(n-1)

x=1e9+rng.normal(0,1,size=200_000)
v_naive=np.mean(x*x)-np.mean(x)**2
mu_w,v_w=welford(x)
print('varianca naive=',v_naive,'Welford=',v_w,'NumPy=',x.var(ddof=1))


## 8. Shembulli i plotë dhe punishte dinamike

Studioni mesataren e një shpërndarjeje eksponenciale. Ndryshoni $N$, numrin e realizimeve $M$ ose shpërndarjen bazë. Vëzhgoni se CLT standardizon mesataret edhe kur të dhënat individuale janë asimetrike. Kur `DINAMIK=True`, histogrami rifreskohet me grupe realizimesh.


In [ ]:
def clt_dinamik(N=20,M=3000,bllok=100,DINAMIK=False):
    mes=[]; fig,ax=plt.subplots(figsize=(7,4))
    for fillim in range(0,M,bllok):
        mes.extend(np.mean(rng.exponential(2,size=(bllok,N)),axis=1))
        if DINAMIK:
            ax.clear(); ax.hist(mes,bins=35,density=True,alpha=.75)
            ax.set(title=f'{len(mes)} mesatare, N={N}',xlabel='mesatarja'); plt.pause(.03)
    if not DINAMIK:
        ax.hist(mes,bins=35,density=True,alpha=.75); ax.set(xlabel='mesatarja',title=f'M={M}, N={N}')
    ax.grid(alpha=.3); plt.show()
    return np.array(mes)

mes=clt_dinamik(DINAMIK=False)
print('SE i matur=',mes.std(ddof=1),'SE teorik=',2/np.sqrt(20))


### Kovarianca mund të rrisë ose ulë pacaktueshmërinë

Për $Y=X_1+X_2$,

$$\operatorname{Var}(Y)=\sigma_1^2+\sigma_2^2+2\operatorname{Cov}(X_1,X_2).$$

Korrelacioni pozitiv rrit variancën e shumës; korrelacioni negativ mund ta ulë. Formula që mbledh vetëm katrorët e pacaktueshmërive është e vlefshme vetëm kur kovariancat janë zero ose të papërfillshme.


In [ ]:
sig=1.; M=200000
for rho in [-.8,0,.8]:
    C=np.array([[1,rho],[rho,1.]])
    X=rng.multivariate_normal([0,0],C,size=M); Y=X[:,0]+X[:,1]
    print(f'rho={rho:+.1f}: var e matur={Y.var():.3f}, teorike={2+2*rho:.3f}')


### Mbulimi i një intervali besimi

Një interval 95% i ndërtuar me procedurë të vlefshme mbulon parametrin e vërtetë në rreth 95% të eksperimenteve të përsëritura. Nuk do të thotë se, pasi intervali është llogaritur, parametri fiks ka probabilitet 95% të jetë brenda tij. Eksperimenti i mëposhtëm kontrollon frekuencën e mbulimit për mesataren normale.


In [ ]:
M,N=5000,40
X=rng.normal(2,3,size=(M,N)); means=X.mean(axis=1); se=X.std(axis=1,ddof=1)/np.sqrt(N)
cover=np.mean((means-1.96*se<=2)&(2<=means+1.96*se))
print('mbulimi empirik=',cover,'objektivi≈0.95')


### Gabimi statistikor dhe gabimi sistematik

Rritja e $N$ zvogëlon gabimin standard, por nuk heq një zhvendosje sistematike të aparatit ose një model të gabuar. Nëse të gjitha matjet janë $0.5$ njësi mbi vlerën reale, mesatarja konvergon me shumë saktësi te vlera e gabuar. Raporti duhet t'i ndajë pacaktueshmëritë e rastit nga burimet sistematike.


In [ ]:
for N in [10,100,1000,10000]:
    matje=5.0+0.5+rng.normal(0,1,N)
    print(N,matje.mean(),matje.std(ddof=1)/np.sqrt(N))


### Pyetje për vetëkontroll

1. Cili është dallimi ndërmjet $\sigma$ dhe $\sigma/\sqrt N$?
2. Kur duhet përfshirë kovarianca në përhapjen e pacaktueshmërisë?
3. Pse një interval 95% nuk është një garanci për çdo mostër?
4. Çfarë lloj gabimi nuk zvogëlohet duke rritur $N$?


# Pjesa II — GNU Octave

## 9. Mesatarja kumulative dhe Welford-i

```octave
rand('seed',2026); p=0.3; N=20000;
x=rand(1,N)<p; n=1:N; mk=cumsum(x)./n;
semilogx(n,mk); hold on; plot([1,N],[p,p],'k--'); grid on;

v=1e9+randn(1,200000); n=0; mu=0; M2=0;
for xi=v
    n=n+1; delta=xi-mu; mu=mu+delta/n; M2=M2+delta*(xi-mu);
end
s2=M2/(n-1);
fprintf('varianca Welford = %.12g\n',s2);
```

## 10. Përhapja Monte Carlo

```octave
M=300000; V=10; sV=.2; I=2; sI=.05;
Vs=V+sV*randn(M,1); Is=I+sI*randn(M,1); R=Vs./Is;
fprintf('R = %.6f, sigma_R = %.6f\n',mean(R),std(R));
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = 'import numpy as np\nrng=np.random.default_rng(2026)\nfor N in [100,1000,10000,100000]:\n x=rng.random(N)<.3\n print(N,x.mean(),np.sqrt(.3*.7/N))\n'
script_path = script_dir / 'laboratori_08_probabiliteti_pacaktueshmeria.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_08_probabiliteti_pacaktueshmeria.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Dallova devijimin standard nga gabimi standard.
- [ ] Nxora ligjin $N^{-1/2}$.
- [ ] Përhapa pacaktueshmërinë me dy metoda.
- [ ] Krahasova formulën naive të variancës me Welford-in.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
